# OCR thô hai luồng Hán/Việt trên Colab (một work, một khoảng trang)

Notebook này làm toàn bộ quy trình trong `docs/ocr_runbook.md` cho **một người**:
PDF → ảnh trang → paddlevl (Hán) → Qwen (Việt) → `build_layers` → `check_ocr_complete` → gói `.zip` gửi về.

**Mỗi người chỉ sửa ô 1** (`WORK`, `PAGES`, `MEMBER`, đường dẫn PDF trên Drive). Sau đó *Runtime → Run all*.

| Người | `WORK` | `PAGES` | `MEMBER` |
|---|---|---|---|
| Thành viên 1 | `CPO_18` | `205-544` | `tv1` |
| Thành viên 2 | `CPO_19` | `5-623` | `tv2` |
| Thành viên 3 | `CPO_19` | `624-833` | `tv3` |
| Thành viên 4 | `CPO_20` | `5-806` | `tv4` |

Yêu cầu: runtime **GPU (T4)** (*Runtime → Change runtime type → T4 GPU*), Google Drive còn trống ~2 GB
(thêm ~9 GB nếu lưu model trên Drive).

**Bị ngắt phiên (Colab thường ngắt sau vài giờ)?** Mở lại notebook, *Run all*. Kết quả OCR nằm trên Drive và
mọi bước đều resume (`--skip-done`), trang đã xong không bị OCR lại. Không bao giờ có hai model chạy cùng lúc:
các ô chạy tuần tự.

## 1. Tham số — mỗi người CHỈ sửa ô này

In [ ]:
WORK   = "CPO_19"          # CPO_18 | CPO_19 | CPO_20
PAGES  = "5-623"           # khoảng trang PDF (1-based), đúng như bảng phân công
MEMBER = "tv2"             # mã người chạy, có trong tên gói gửi về

# PDF nguồn trên Drive (đúng bản nhóm phát; SHA-256 được kiểm tra với config)
PDF_ON_DRIVE = "/content/drive/MyDrive/han_viet_ocr/NgoThiNhamToanTap3_LamGiang.pdf"

# Thư mục làm việc trên Drive: lưu 03_ocr (kết quả), log, gói zip -> không mất khi phiên Colab bị ngắt
DRIVE_ROOT = "/content/drive/MyDrive/han_viet_ocr"

# Mã nguồn: git (mọi người dùng CÙNG một tag/commit) hoặc một file zip của repo trên Drive
REPO_URL = "https://github.com/khangthinhne/chinese-vietnamese-corpus.git"
GIT_REF  = "ocr-v1"        # tag/branch/commit do người điều phối thông báo
REPO_ZIP = ""              # vd "/content/drive/MyDrive/han_viet_ocr/repo.zip" (bỏ trống = dùng git)
# Repo private: thêm secret GITHUB_TOKEN trong Colab (biểu tượng chìa khoá bên trái), KHÔNG dán token vào đây.

SEED_ZIP = ""              # gói hạt giống (trang đã OCR sẵn), vd ".../CPO_19_0015-0121_seed_20261125.zip"
HF_CACHE_ON_DRIVE = False  # True: lưu model (~9 GB) trên Drive, phiên sau khỏi tải lại
KEEP_IMAGES_ON_DRIVE = False  # True: lưu ảnh trang trên Drive (~1-2 MB/trang); False: trích lại mỗi phiên (vài phút)

## 2. Kiểm tra GPU

In [ ]:
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit("KHÔNG có GPU. Runtime -> Change runtime type -> T4 GPU, rồi chạy lại.")
print(out.stdout)

## 3. Drive, thư mục, hàm chạy lệnh

In [ ]:
import os, re, json, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

assert re.fullmatch(r"CPO_\d+", WORK), "WORK dạng CPO_xx"
assert re.fullmatch(r"\d+(-\d+)?(,\d+(-\d+)?)*", PAGES), "PAGES dạng 5-623"
REPO = Path("/content/han-viet-corpus")
DRIVE = Path(DRIVE_ROOT); (DRIVE / "logs").mkdir(parents=True, exist_ok=True); (DRIVE / "dist").mkdir(exist_ok=True)
VENV_PADDLE, VENV_QWEN = Path("/content/venv-paddle"), Path("/content/venv-qwen")
PADDLE_PY, QWEN_PY = VENV_PADDLE / "bin/python", VENV_QWEN / "bin/python"
HF_CACHE = (DRIVE / "hf_cache") if HF_CACHE_ON_DRIVE else Path("/content/hf_cache")

def run(cmd, cwd=None, env=None, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> dừng notebook (trừ khi check=False)."""
    print(">>", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    full_env = {**os.environ, "PYTHONUTF8": "1", "HF_HUB_CACHE": str(HF_CACHE), **(env or {})}
    p = subprocess.Popen(cmd, cwd=cwd, env=full_env, shell=isinstance(cmd, str),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if check and code != 0:
        raise SystemExit(f"Lệnh lỗi (mã {code}). Xem output ở trên; bảng xử lý lỗi: docs/ocr_runbook.md mục 5")
    return code

## 4. Lấy mã nguồn (đúng tag/commit chung của nhóm)

In [ ]:
if REPO_ZIP:
    if not REPO.exists():
        tmp = Path("/content/repo_unzip"); shutil.rmtree(tmp, ignore_errors=True)
        shutil.unpack_archive(REPO_ZIP, tmp)
        top = [p for p in tmp.iterdir() if p.is_dir()]
        shutil.move(str(top[0] if len(top) == 1 and not (tmp / "src").exists() else tmp), REPO)
else:
    url = REPO_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass
    if not (REPO / ".git").exists():
        run(["git", "clone", "--quiet", url, str(REPO)])
    run(["git", "fetch", "--quiet", "--tags", "origin"], cwd=REPO, check=False)
    run(["git", "checkout", "--quiet", GIT_REF], cwd=REPO)
    run(["git", "log", "-1", "--format=commit %H (%d) %s"], cwd=REPO)
assert (REPO / "configs" / f"{WORK}.json").exists(), f"Không có configs/{WORK}.json trong repo"

## 5. Cài đặt: Poppler + HAI môi trường riêng
- `venv-paddle`: paddlevl (transformers ≥ 5, torch ≥ 2.6)
- `venv-qwen`: Qwen2.5-VL 4-bit + mọi bước còn lại (transformers 4.57.6, giống máy nhóm)

Không dùng Python sẵn của Colab cho OCR (phiên bản thư viện đổi theo Colab). Lần đầu mất ~5-10 phút.

In [ ]:
run("apt-get -qq update && apt-get -qq install -y poppler-utils > /dev/null")
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
COMMON_PKGS = ["python-dotenv", "pillow", "numpy", "pdf2image", "opencc", "huggingface_hub", "pytest"]
if not PADDLE_PY.exists():
    run(["uv", "venv", "-q", "--python", "3.11", str(VENV_PADDLE)])
    run(["uv", "pip", "install", "-q", "--python", str(PADDLE_PY), "torch==2.14.1",
         "--index-url", "https://download.pytorch.org/whl/cu126"])
    run(["uv", "pip", "install", "-q", "--python", str(PADDLE_PY), "transformers==5.24.0", *COMMON_PKGS])
if not QWEN_PY.exists():
    run(["uv", "venv", "-q", "--python", "3.11", str(VENV_QWEN)])
    run(["uv", "pip", "install", "-q", "--python", str(QWEN_PY), "torch==2.5.1", "torchvision==0.20.1",
         "--index-url", "https://download.pytorch.org/whl/cu121"])
    run(["uv", "pip", "install", "-q", "--python", str(QWEN_PY), "transformers==4.57.6", "accelerate==1.2.1",
         "bitsandbytes==0.50.2", "qwen-vl-utils", *COMMON_PKGS])
for py in (PADDLE_PY, QWEN_PY):
    run([str(py), "-c", "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"])

## 6. Dữ liệu: kết quả OCR lưu trên Drive, PDF nguồn, kiểm tra SHA-256
`data/<WORK>/03_ocr` là symlink tới Drive. `raw/*.json` chỉ được thêm run, không bao giờ bị sửa/xoá.

In [ ]:
work_dir = REPO / "data" / WORK
work_dir.mkdir(parents=True, exist_ok=True)
links = {"03_ocr": DRIVE / WORK / "03_ocr"}
if KEEP_IMAGES_ON_DRIVE:
    links["02_extraction"] = DRIVE / WORK / "02_extraction"
for name, target in links.items():
    target.mkdir(parents=True, exist_ok=True)
    local = work_dir / name
    if local.is_symlink():
        continue
    if local.exists():                       # repo có sẵn vài file đã commit: chép sang Drive (không ghi đè), rồi link
        for f in local.rglob("*"):
            dst = target / f.relative_to(local)
            if f.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
        shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)
logs = REPO / "logs"
if not logs.is_symlink():
    shutil.rmtree(logs, ignore_errors=True); logs.symlink_to(DRIVE / "logs", target_is_directory=True)

cfg = json.loads((REPO / "configs" / f"{WORK}.json").read_text(encoding="utf-8"))
source = REPO / cfg["common"]["source"]
if not source.exists():
    assert Path(PDF_ON_DRIVE).exists(), f"Không thấy PDF trên Drive: {PDF_ON_DRIVE}"
    source.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PDF_ON_DRIVE, source)
run([str(QWEN_PY), "-m", "src.ocr.work_config", "--config", f"configs/{WORK}.json", "--verify-source"], cwd=REPO)
run(["pdfinfo", str(source)], check=False)

## 7. Tải model (một lần mỗi phiên, hoặc một lần nếu `HF_CACHE_ON_DRIVE=True`), rồi chạy offline

In [ ]:
for py, model in ((PADDLE_PY, "PaddlePaddle/PaddleOCR-VL-1.6"), (QWEN_PY, "Qwen/Qwen2.5-VL-3B-Instruct")):
    run([str(py), "-c", f"from huggingface_hub import snapshot_download as s; print(s('{model}'))"],
        env={"HF_HUB_OFFLINE": "0"})
os.environ["HF_HUB_OFFLINE"] = "1"

## 8. (Tuỳ chọn) Giải gói hạt giống — trang đã OCR sẵn sẽ được bỏ qua

In [ ]:
if SEED_ZIP:
    run([str(QWEN_PY), "scripts/unpack_ocr.py", SEED_ZIP, "--data-root", "data"], cwd=REPO, check=False)
else:
    print("Không có gói hạt giống.")

## 9. Kiểm tra sẵn sàng + chạy khô (không nạp model): trang nào đã xong, trang nào sẽ OCR, ước lượng thời gian

In [ ]:
ENV = {"PADDLE_PY": str(PADDLE_PY), "QWEN_PY": str(QWEN_PY), "DATA_ROOT": "data", "HF_HUB_OFFLINE": "1"}
run([str(PADDLE_PY), "-m", "src.ocr.preflight", "--engine", "paddlevl"], cwd=REPO, env=ENV)
run([str(QWEN_PY), "-m", "src.ocr.preflight", "--engine", "qwen"], cwd=REPO, env=ENV)
run(["bash", "scripts/ocr_work.sh", WORK, PAGES], cwd=REPO,
    env={**ENV, "DRY_RUN": "1", "STEPS": "han vie"})

## 10. Trích ảnh trang (chỉ các trang trong `PAGES`, ảnh đã có thì bỏ qua)

In [ ]:
run(["bash", "scripts/ocr_work.sh", WORK, PAGES], cwd=REPO, env={**ENV, "STEPS": "extract"})

## 11. Luồng Hán — paddlevl (T4: tự dùng float32; ~25-40 s/trang, chưa đo trên T4)
Bị ngắt giữa chừng: chạy lại ô này (hoặc *Run all*), trang đã xong được bỏ qua. Log: `DRIVE_ROOT/logs/`.

In [ ]:
run(["bash", "scripts/ocr_work.sh", WORK, PAGES], cwd=REPO, env={**ENV, "STEPS": "han"})

## 12. Luồng Việt — Qwen2.5-VL-3B 4-bit (~30-50 s/trang trên T4, chưa đo)

In [ ]:
run(["bash", "scripts/ocr_work.sh", WORK, PAGES], cwd=REPO, env={**ENV, "STEPS": "vie"})

## 13. Dựng han.xml / vie.xml / TXT và kiểm tra trong `PAGES`
"ĐỦ" = 0 lỗi. Cảnh báo cần người xem (bảng mã ở `docs/ocr_runbook.md` mục 6). In thêm 5 trang ngẫu nhiên để
soát bằng mắt: mở ảnh `data/<WORK>/02_extraction/pages/page_NNN.png` và so với `03_ocr/ocr_raw_text/{han,vie}/NNNN.txt`.

In [ ]:
code = run(["bash", "scripts/ocr_work.sh", WORK, PAGES], cwd=REPO, env={**ENV, "STEPS": "build check"}, check=False)
print("\nKẾT QUẢ:", "ĐỦ - có thể đóng gói" if code == 0 else "CHƯA ĐỦ - xem lỗi ở trên; thường chỉ cần chạy lại ô 11/12")

## 14. Đóng gói và lưu lên Drive (gửi file này cho người điều phối)

In [ ]:
run([str(QWEN_PY), "scripts/pack_ocr.py", "--work-id", WORK, "--pages", PAGES, "--member", MEMBER,
     "--data-root", "data", "--out", str(DRIVE / "dist"), "--force"], cwd=REPO)
for z in sorted((DRIVE / "dist").glob(f"{WORK}_*_{MEMBER}_*.zip")):
    print(f"{z}  ({z.stat().st_size / 1e6:.1f} MB)")
print("Gửi file .zip ở trên qua Drive/Teams. KHÔNG gửi .env, không commit data/ hay dist/.")

## Ghi chú
- **Mọi người dùng cùng `GIT_REF`.** Commit, GPU, dtype được ghi vào từng run (`provenance`). T4 dùng float32 cho
  paddlevl nên khoá cache khác máy RTX 3050 (bf16). Điều này không phải lỗi: `--skip-done` bỏ qua trang theo
  số trang, và khi ghép, `build_layers` chọn run theo manifest, rồi run hợp lệ, rồi run mới nhất.
- Xử lý lỗi: `docs/ocr_runbook.md` mục 5. Hay gặp nhất: OOM thì script tự thử lại với `*_MAX_PIXELS_TOKENS=768`;
  mất kết nối thì *Run all* lại.
- Chạy một trang bằng tay (vd OCR lại trang 208 với dải thấp hơn):
  `QWEN_BAND_INCHES=1.0 /content/venv-qwen/bin/python -m src.ocr.ocr --config configs/CPO_18.json --layer vie --engines qwen --pages 208`